# ルールを書く vs 例から学ばせる

**出典**　機械学習 4 分野の教科書 — 第 1 章 総論 ／ 01 直感：迷惑メールのフィルタを作ってみる

**このノートのパス**　`ml-code/01_overview/01_rule_vs_learning.ipynb`

**このノートで確かめること**
同じ「スパム判定」を、①人間がルールを書く方法と ②例から学ばせる方法の両方で書き、
**ルールを保守する作業が、データを集める作業に置き換わる**ことを体感する。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `scikit-learn` |
| 実行時間の目安 | 約 10 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [1]:
# 未導入なら実行する（Colab など）
# !pip install -q scikit-learn

## ① 従来のプログラミング — ルールを人間が書く

条件を思いつく限り並べる。**新しい手口が出るたびに、人間が行を足す**ことになる。

In [2]:
def is_spam(mail: str) -> bool:
    if "無料" in mail and "今すぐ" in mail:
        return True
    if mail.count("!") > 5:
        return True
    if "クリック" in mail and "http" in mail:
        return True
    return False


for m in ["無料で今すぐ登録！", "明日の会議は10時からです", "限定オファー！今だけ"]:
    print(f"{is_spam(m)!s:>5}  {m}")

 True  無料で今すぐ登録！
False  明日の会議は10時からです
False  限定オファー！今だけ


👆 3 つめの「限定オファー！今だけ」が **False** になる。
「無料」も「クリック」も入っていないからである。**ルールを 1 本足せば直るが、その繰り返しが続く。**

## ② 機械学習 — ルールは学習して得る

`analyzer="char_wb"` は**文字 N-gram**。日本語は単語が空白で区切られないので、
既定の単語分割（空白区切り）では文全体が 1 語になってしまう。ここを間違えると精度が出ない。

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

mails = ["無料で今すぐ登録！", "限定キャンペーン実施中", "副収入を保証します",
         "明日の会議は10時からです", "資料を添付しました", "来週の出張の件です"]
labels = [1, 1, 1, 0, 0, 0]        # 正解（1=スパム, 0=通常）

# analyzer="char_wb" ＝ 文字 N-gram。日本語には空白区切りが無いので必須
model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    LogisticRegression(max_iter=1000),
)
model.fit(mails, labels)          # ← ここで「規則」が作られる

print(model.predict(["限定オファー！今だけ"]))        # → [1]  スパム
print(model.predict(["明日の打ち合わせの資料です"]))   # → [0]  通常

# 予測確率 [通常, スパム]。6 通しか学習していないので、自信はほとんど無い
print(model.predict_proba(["限定オファー！今だけ", "明日の打ち合わせの資料です"]).round(2))

[1]
[0]
[[0.46 0.54]
 [0.55 0.45]]


## 違いがどこに出たか

| | ルールベース | 機械学習 |
|---|---|---|
| 「限定オファー！今だけ」 | **False**（取りこぼす） | **1**（スパムと判定） |
| 新しい手口への対応 | 人間が `if` を足す | **例を足して学習し直す** |
| 判断根拠 | コードを読めば分かる | 係数を見る必要がある |

⚠️ 学習データが 6 件しかないので、**この結果を汎化性能と読んではいけない**。
ここで確かめたのは精度ではなく、**保守の対象がコードからデータに移る**という構造の違いである。